In [1]:
# 第1步：使用本地上传的YOLOv5而不是克隆
print("步骤1: 设置YOLOv5环境")

# 检查上传的YOLOv5文件夹
import os
import shutil
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from glob import glob
from tqdm import tqdm
import torch
from sklearn.model_selection import train_test_split

# 定义YOLOv5路径 - 这里需要根据你上传的数据集路径进行调整
# 假设上传后的路径为 /kaggle/input/yolov5-local/
YOLOV5_PATH = '/kaggle/input/yolov5/pytorch/default/3/yolov5-6.0'

# 如果需要，复制到工作目录以便有写入权限
if os.path.exists(YOLOV5_PATH):
    print(f"找到上传的YOLOv5文件夹: {YOLOV5_PATH}")
    
    # 创建工作目录中的yolov5文件夹
    os.makedirs('/kaggle/working/yolov5', exist_ok=True)
    
    # 复制必要文件（可以根据需要调整）
    important_files = ['requirements.txt', 'train.py', 'detect.py', 'export.py', 'val.py']
    for file in important_files:
        src = os.path.join(YOLOV5_PATH, file)
        dst = os.path.join('/kaggle/working/yolov5', file)
        if os.path.exists(src):
            shutil.copy(src, dst)
            print(f"复制文件: {file}")
    
    # 复制必要的文件夹
    important_dirs = ['models', 'utils', 'data']
    for dir_name in important_dirs:
        src_dir = os.path.join(YOLOV5_PATH, dir_name)
        dst_dir = os.path.join('/kaggle/working/yolov5', dir_name)
        if os.path.exists(src_dir):
            shutil.copytree(src_dir, dst_dir)
            print(f"复制文件夹: {dir_name}")
    
    # 进入YOLOv5目录
    %cd /kaggle/working/yolov5
    
    # 安装依赖
    !pip install -r requirements.txt
    
    # 回到工作目录
    %cd /kaggle/working
    
else:
    print("未找到上传的YOLOv5文件夹，请确保已正确上传")
    print("请在右侧'Add data'菜单中上传本地YOLOv5文件夹")

步骤1: 设置YOLOv5环境
找到上传的YOLOv5文件夹: /kaggle/input/yolov5/pytorch/default/12/yolov5-6.0
复制文件: requirements.txt
复制文件: train.py
复制文件: detect.py
复制文件: export.py
复制文件: val.py
复制文件夹: models
复制文件夹: utils
复制文件夹: data
/kaggle/working/yolov5
/kaggle/working


In [2]:
# 定义常量和路径 - 注意这里不再默认/data目录
INPUT_DIR = '/kaggle/input'  # 根据实际路径调整
WORKING_DIR = '/kaggle/working'
OUTPUT_DIR = f'{WORKING_DIR}/NIH_ChestXray_YOLO'

In [3]:
# 第2步：找到所有图像文件
print("步骤2: 查找图像文件")

# 查找所有图像文件夹 - 直接检查全部输入目录
image_folders = sorted(glob(f'{INPUT_DIR}/*/images_*'))
print(f"找到 {len(image_folders)} 个图像文件夹: {image_folders}")

# 查找所有PNG图像
all_images = []
for folder in image_folders:
    images = glob(f'{folder}/images/*.png')
    all_images.extend(images)
    print(f"在 {os.path.basename(folder)} 中找到 {len(images)} 张图像")

print(f"总共找到 {len(all_images)} 张图像")

# 创建一个映射，从图像名到完整路径
image_path_map = {os.path.basename(img): img for img in all_images}

步骤2: 查找图像文件
找到 12 个图像文件夹: ['/kaggle/input/data/images_001', '/kaggle/input/data/images_002', '/kaggle/input/data/images_003', '/kaggle/input/data/images_004', '/kaggle/input/data/images_005', '/kaggle/input/data/images_006', '/kaggle/input/data/images_007', '/kaggle/input/data/images_008', '/kaggle/input/data/images_009', '/kaggle/input/data/images_010', '/kaggle/input/data/images_011', '/kaggle/input/data/images_012']
在 images_001 中找到 4999 张图像
在 images_002 中找到 10000 张图像
在 images_003 中找到 10000 张图像
在 images_004 中找到 10000 张图像
在 images_005 中找到 10000 张图像
在 images_006 中找到 10000 张图像
在 images_007 中找到 10000 张图像
在 images_008 中找到 10000 张图像
在 images_009 中找到 10000 张图像
在 images_010 中找到 10000 张图像
在 images_011 中找到 10000 张图像
在 images_012 中找到 7121 张图像
总共找到 112120 张图像


In [4]:
# 第3步：数据转换函数实现
print("步骤3: 定义数据转换函数")

def extract_disease_classes_from_bbox(bbox_file):
    """从边界框数据中提取实际标注的疾病类别"""
    bbox_df = pd.read_csv(bbox_file)
    unique_diseases = bbox_df['Finding Label'].unique()
    return sorted(list(unique_diseases))

def convert_bbox_to_yolo(image_width, image_height, bbox):
    """
    将[x, y, width, height]格式的边界框转换为YOLOv5格式[x_center, y_center, width, height]
    其中所有值都相对于图像尺寸进行归一化
    """
    x, y, w, h = bbox
    # 计算中心点坐标
    x_center = (x + w/2) / image_width
    y_center = (y + h/2) / image_height
    # 归一化宽度和高度
    w_norm = w / image_width
    h_norm = h / image_height
    
    return [x_center, y_center, w_norm, h_norm]

def create_custom_dataset_split(bbox_file, image_path_map, train_ratio=0.7, val_ratio=0.15, test_ratio=0.15, random_seed=42):
    """
    创建自定义数据集划分，确保所有包含边界框的图像都被合理分配
    
    参数:
        bbox_file: 边界框数据文件路径
        image_path_map: 图像名称到路径的映射字典
        train_ratio, val_ratio, test_ratio: 训练、验证、测试集的比例
        random_seed: 随机种子，确保结果可复现
    
    返回:
        train_imgs, val_imgs, test_imgs: 包含图像名称的列表
    """
    import pandas as pd
    import numpy as np
    from sklearn.model_selection import train_test_split
    
    # 读取边界框数据
    bbox_df = pd.read_csv(bbox_file)
    
    # 获取所有有边界框的唯一图像
    unique_images = bbox_df['Image Index'].unique()
    
    # 过滤，仅保留在image_path_map中存在的图像
    available_images = [img for img in unique_images if img in image_path_map]
    print(f"找到 {len(available_images)}/{len(unique_images)} 张有边界框的图像")
    
    # 确保有足够数量的图像
    if len(available_images) < 10:
        raise ValueError("可用图像数量过少，无法进行有效的数据集划分")
    
    # 统计每个图像的疾病标签数量
    image_label_counts = bbox_df.groupby('Image Index')['Finding Label'].nunique().to_dict()
    
    # 按照标签数量和疾病类型分层，确保各个集合中的疾病分布相似
    disease_labels = bbox_df.groupby('Image Index')['Finding Label'].apply(list).to_dict()
    
    # 先根据标签数量将图像分组
    label_count_groups = {}
    for img in available_images:
        count = image_label_counts.get(img, 0)
        if count not in label_count_groups:
            label_count_groups[count] = []
        label_count_groups[count].append(img)
    
    # 分割测试集和剩余数据
    np.random.seed(random_seed)
    train_val_imgs, test_imgs = [], []
    
    for count, imgs in label_count_groups.items():
        n_test = max(1, int(len(imgs) * test_ratio))
        test_from_group = np.random.choice(imgs, n_test, replace=False).tolist()
        train_val_from_group = [img for img in imgs if img not in test_from_group]
        
        test_imgs.extend(test_from_group)
        train_val_imgs.extend(train_val_from_group)
    
    # 分割训练集和验证集
    val_ratio_adjusted = val_ratio / (train_ratio + val_ratio)  # 调整验证集比例
    train_imgs, val_imgs = train_test_split(
        train_val_imgs,
        test_size=val_ratio_adjusted,
        random_state=random_seed
    )
    
    print(f"数据集划分: {len(train_imgs)} 训练, {len(val_imgs)} 验证, {len(test_imgs)} 测试")
    
    # 确保每个疾病类别在各个集合中都有代表
    disease_classes = bbox_df['Finding Label'].unique()
    
    for disease in disease_classes:
        # 找出含有此疾病的图像
        disease_images = bbox_df[bbox_df['Finding Label'] == disease]['Image Index'].unique()
        disease_images = [img for img in disease_images if img in image_path_map]
        
        # 检查此疾病是否在各个集合中都有代表
        train_has = any(img in train_imgs for img in disease_images)
        val_has = any(img in val_imgs for img in disease_images)
        test_has = any(img in test_imgs for img in disease_images)
        
        # 如果某个集合中没有此疾病，则从其他集合中移动一个样本
        if not train_has and len(disease_images) > 0:
            for img in disease_images:
                if img in val_imgs:
                    val_imgs.remove(img)
                    train_imgs.append(img)
                    break
                elif img in test_imgs:
                    test_imgs.remove(img)
                    train_imgs.append(img)
                    break
        
        if not val_has and len(disease_images) > 0:
            for img in disease_images:
                if img in train_imgs:
                    train_imgs.remove(img)
                    val_imgs.append(img)
                    break
        
        if not test_has and len(disease_images) > 0:
            for img in disease_images:
                if img in train_imgs and len(train_imgs) > 1:
                    train_imgs.remove(img)
                    test_imgs.append(img)
                    break
    
    # 打印每个集合中的疾病分布
    print("\n各集合中的疾病分布:")
    for dataset_name, dataset in [("训练集", train_imgs), ("验证集", val_imgs), ("测试集", test_imgs)]:
        dataset_diseases = []
        for img in dataset:
            if img in disease_labels:
                dataset_diseases.extend(disease_labels[img])
        
        disease_counts = {}
        for disease in dataset_diseases:
            if disease in disease_counts:
                disease_counts[disease] += 1
            else:
                disease_counts[disease] = 1
        
        print(f"{dataset_name}: {disease_counts}")
    
    return train_imgs, val_imgs, test_imgs

def convert_data(bbox_file, output_dir, image_path_map, image_width=1024, image_height=1024):
    """
    转换NIH胸部X光数据集为YOLOv5格式，使用自定义的数据集划分
    """
    # 读取边界框数据
    bbox_df = pd.read_csv(bbox_file)
    
    # 提取边界框数据中的疾病类别
    disease_classes = extract_disease_classes_from_bbox(bbox_file)
    print(f"从BBox_List_2017.csv中提取的疾病类别: {disease_classes}")
    
    # 创建疾病到类别ID的映射
    class_mapping = {disease: idx for idx, disease in enumerate(disease_classes)}
    
    # 创建输出目录
    os.makedirs(os.path.join(output_dir, 'labels'), exist_ok=True)
    os.makedirs(os.path.join(output_dir, 'images'), exist_ok=True)
    
    # 使用自定义划分函数
    train_imgs, val_imgs, test_imgs = create_custom_dataset_split(
        bbox_file=bbox_file,
        image_path_map=image_path_map,
        train_ratio=0.7,
        val_ratio=0.15,
        test_ratio=0.15,
        random_seed=42
    )
    
    # 创建data.yaml文件
    with open(os.path.join(output_dir, 'data.yaml'), 'w') as f:
        f.write(f"# YOLOv5数据集配置\n")
        f.write(f"path: {output_dir}\n")
        f.write(f"train: images/train\n")
        f.write(f"val: images/val\n")
        f.write(f"test: images/test\n\n")
        f.write(f"# 类别\n")
        f.write(f"nc: {len(disease_classes)}\n")
        f.write(f"names: {disease_classes}\n")
    
    # 创建数据集目录
    for split, imgs in zip(['train', 'val', 'test'], [train_imgs, val_imgs, test_imgs]):
        if imgs:  # 只创建非空目录
            os.makedirs(os.path.join(output_dir, 'images', split), exist_ok=True)
            os.makedirs(os.path.join(output_dir, 'labels', split), exist_ok=True)
    
    # 处理每个图像
    processed_images = 0
    for img_file in tqdm(train_imgs + val_imgs + test_imgs, desc="处理图像"):
        # 获取所有边界框
        img_bboxes = bbox_df[bbox_df['Image Index'] == img_file]
        
        # 确定图像应该放在哪个集合中
        if img_file in train_imgs:
            split = 'train'
        elif img_file in val_imgs:
            split = 'val'
        elif img_file in test_imgs:
            split = 'test'
        else:
            # 跳过不在任何划分中的图像
            continue
        
        # 创建标签文件
        label_file = os.path.join(output_dir, 'labels', split, img_file.replace('.png', '.txt'))
        
        with open(label_file, 'w') as f:
            for _, row in img_bboxes.iterrows():
                disease = row['Finding Label']
                if disease in class_mapping:
                    class_id = class_mapping[disease]
                    # 确保边界框数据有效
                    try:
                        bbox = [float(row['Bbox [x']), float(row['y']), float(row['w']), float(row['h]'])]
                        if all(v >= 0 for v in bbox) and bbox[2] > 0 and bbox[3] > 0:  # 有效边界框
                            yolo_bbox = convert_bbox_to_yolo(image_width, image_height, bbox)
                            f.write(f"{class_id} {' '.join([str(x) for x in yolo_bbox])}\n")
                    except (ValueError, TypeError) as e:
                        print(f"跳过无效边界框: {row}, 错误: {e}")
        
        # 复制或链接图像文件
        src_img = image_path_map[img_file]
        dst_img = os.path.join(output_dir, 'images', split, img_file)
        
        # 检查目标文件是否已存在，如果存在则跳过
        if os.path.exists(dst_img):
            # 文件已存在，跳过
            pass
        else:
            try:
                # 尝试创建符号链接
                os.symlink(src_img, dst_img)
            except OSError:
                # 如果符号链接失败，则尝试复制文件
                try:
                    shutil.copy(src_img, dst_img)
                except shutil.SameFileError:
                    # 如果是同一个文件，可能是因为路径解析问题，跳过
                    print(f"警告: 源文件和目标文件相同 - {src_img}")
        
        processed_images += 1
    
    print(f"数据集转换完成！处理了 {processed_images} 张图像")
    print(f"训练集大小: {len(train_imgs)}")
    print(f"验证集大小: {len(val_imgs)}")
    print(f"测试集大小: {len(test_imgs)}")
    print(f"目标疾病类别: {disease_classes}")
    
    return disease_classes if processed_images > 0 else []

步骤3: 定义数据转换函数


In [5]:
# 第4步：检查输入数据存在性并转换数据
print("步骤4: 检查输入数据并准备数据集")

# 尝试不同可能的路径查找BBox文件
bbox_file_candidates = [
    f'{INPUT_DIR}/BBox_List_2017.csv',
    f'{INPUT_DIR}/data/BBox_List_2017.csv'
]

# 查找文件
bbox_file = None
for candidate in bbox_file_candidates:
    if os.path.exists(candidate):
        bbox_file = candidate
        print(f"✓ 找到文件: {bbox_file}")
        break

# 同样尝试查找训练测试划分文件
train_val_list = None
test_list = None
for base_dir in [INPUT_DIR, f'{INPUT_DIR}/data']:
    if os.path.exists(f'{base_dir}/train_val_list.txt'):
        train_val_list = f'{base_dir}/train_val_list.txt'
        print(f"✓ 找到文件: {train_val_list}")
    
    if os.path.exists(f'{base_dir}/test_list.txt'):
        test_list = f'{base_dir}/test_list.txt'
        print(f"✓ 找到文件: {test_list}")

# 如果没有找到文件，打印错误并显示目录结构
if not bbox_file:
    print("✗ 未找到BBox_List_2017.csv文件")
    print("列出INPUT_DIR下的文件:")
    !ls -la {INPUT_DIR}
    
    # 如果有data子目录，也列出
    if os.path.exists(f'{INPUT_DIR}/data'):
        print("列出INPUT_DIR/data下的文件:")
        !ls -la {INPUT_DIR}/data

# 转换数据
if bbox_file and len(image_path_map) > 0:
    disease_classes = convert_data(
        bbox_file=bbox_file,
        output_dir=OUTPUT_DIR,
        image_path_map=image_path_map,
    )
else:
    print("数据文件缺失，无法继续")
    disease_classes = []

# 确认数据集结构
if os.path.exists(OUTPUT_DIR) and disease_classes:
    print("\n数据集结构:")
    train_dir = os.path.join(OUTPUT_DIR, 'labels', 'train')
    if os.path.exists(train_dir) and len(os.listdir(train_dir)) > 0:
        !ls -la {train_dir} | head -5
        print("...")
    
    images_train_dir = os.path.join(OUTPUT_DIR, 'images', 'train')
    if os.path.exists(images_train_dir) and len(os.listdir(images_train_dir)) > 0:
        !ls -la {images_train_dir} | head -5
        print("...")
    
    !cat {OUTPUT_DIR}/data.yaml

步骤4: 检查输入数据并准备数据集
✓ 找到文件: /kaggle/input/data/BBox_List_2017.csv
✓ 找到文件: /kaggle/input/data/train_val_list.txt
✓ 找到文件: /kaggle/input/data/test_list.txt
从BBox_List_2017.csv中提取的疾病类别: ['Atelectasis', 'Cardiomegaly', 'Effusion', 'Infiltrate', 'Mass', 'Nodule', 'Pneumonia', 'Pneumothorax']
找到 880/880 张有边界框的图像
数据集划分: 616 训练, 132 验证, 132 测试

各集合中的疾病分布:
训练集: {'Cardiomegaly': 111, 'Infiltrate': 85, 'Pneumonia': 89, 'Atelectasis': 121, 'Nodule': 52, 'Mass': 61, 'Pneumothorax': 64, 'Effusion': 105}
验证集: {'Pneumonia': 12, 'Infiltrate': 20, 'Cardiomegaly': 19, 'Effusion': 27, 'Atelectasis': 30, 'Nodule': 9, 'Pneumothorax': 17, 'Mass': 13}
测试集: {'Nodule': 18, 'Atelectasis': 29, 'Cardiomegaly': 16, 'Effusion': 21, 'Infiltrate': 18, 'Mass': 11, 'Pneumothorax': 17, 'Pneumonia': 19}


处理图像: 100%|██████████| 880/880 [00:00<00:00, 1494.42it/s]


数据集转换完成！处理了 880 张图像
训练集大小: 616
验证集大小: 132
测试集大小: 132
目标疾病类别: ['Atelectasis', 'Cardiomegaly', 'Effusion', 'Infiltrate', 'Mass', 'Nodule', 'Pneumonia', 'Pneumothorax']

数据集结构:
total 2492
drwxr-xr-x 2 root root 24576 Feb 27 03:43 .
drwxr-xr-x 5 root root  4096 Feb 27 03:43 ..
-rw-r--r-- 1 root root    78 Feb 27 03:43 00000032_037.txt
-rw-r--r-- 1 root root    80 Feb 27 03:43 00000149_006.txt
ls: write error: Broken pipe
...
total 24
drwxr-xr-x 2 root root 20480 Feb 27 03:43 .
drwxr-xr-x 5 root root  4096 Feb 27 03:43 ..
lrwxrwxrwx 1 root root    53 Feb 27 03:43 00000032_037.png -> /kaggle/input/data/images_001/images/00000032_037.png
lrwxrwxrwx 1 root root    53 Feb 27 03:43 00000149_006.png -> /kaggle/input/data/images_001/images/00000149_006.png
ls: write error: Broken pipe
...
# YOLOv5数据集配置
path: /kaggle/working/NIH_ChestXray_YOLO
train: images/train
val: images/val
test: images/test

# 类别
nc: 8
names: ['Atelectasis', 'Cardiomegaly', 'Effusion', 'Infiltrate', 'Mass', 'Nodule', 'Pneumon

In [6]:
# 第5步：训练YOLOv5模型
print("步骤5: 训练YOLOv5模型")

if disease_classes and os.path.exists(OUTPUT_DIR):
    print(f"开始训练模型，数据集路径: {OUTPUT_DIR}")
    
    # 设置训练参数
    BATCH_SIZE = 16
    EPOCHS = 10  # 可以根据需要调整
    IMAGE_SIZE = 640
    MODEL_TYPE = 's'  # 可选 'n', 's', 'm', 'l', 'x'
    
    # 执行训练命令
    %cd /kaggle/working/yolov5
    !python train.py --img {IMAGE_SIZE} --batch {BATCH_SIZE} --epochs {EPOCHS} \
                     --data {OUTPUT_DIR}/data.yaml \
                     --cfg models/yolov5{MODEL_TYPE}.yaml \
                     --weights yolov5{MODEL_TYPE}.pt \
                     --name chest_xray_model \
                     --cache

    # # 增加训练轮数
    # EPOCHS = 5  # 原为10
    
    # # 减小批次大小以适应更复杂的模型
    # BATCH_SIZE = 8  # 原为16
    
    # # 使用更大的图像尺寸以获取更详细的特征
    # IMAGE_SIZE = 800  # 原为640
    
    # # 考虑使用更大的模型
    # MODEL_TYPE = 's'  # 使用medium模型替代small('s')
    
    # # 添加额外参数提高多标签处理能力
    # %cd /kaggle/working/yolov5
    # !python train.py --img {IMAGE_SIZE} --batch {BATCH_SIZE} --epochs {EPOCHS} \
    #                  --data {OUTPUT_DIR}/data.yaml \
    #                  --cfg models/yolov5{MODEL_TYPE}.yaml \
    #                  --weights yolov5{MODEL_TYPE}.pt \
    #                  --name chest_xray_model \
    #                  --multi-label \
    #                  --label-smoothing 0.1 \
    #                  --cache
    
    # 回到工作目录
    %cd /kaggle/working
    
    print("训练完成！")
else:
    print("数据集准备不完整，无法开始训练")

步骤5: 训练YOLOv5模型
开始训练模型，数据集路径: /kaggle/working/NIH_ChestXray_YOLO
/kaggle/working/yolov5
2025-02-27 03:43:49.629921: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-02-27 03:43:49.821253: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-02-27 03:43:49.879194: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice: (30 second timeout) 
wandb: W&B